#  GCN Implementation


The GCN baseline follows Kipf & Welling (2017): two `GCNConv` layers with a hidden dimension of 16, ReLU activation between layers, and dropout of 0.5 for regularization. `cached=True` caches the normalized adjacency matrix after the first forward pass so it is not recomputed every epoch. Unlike SGC, the graph propagation happens inside the forward pass on every training step, meaning gradients flow back through the graph structure each epoch. This is what makes GCN slower and more memory intensive than SGC especially on large graphs like Reddit where it runs out of memory entirely.

### Imports/Installations

In [ ]:
%%capture
!pip install torch torchvision torchaudio
!pip install torch_geometric

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### GCN Implementation

In [ ]:
class GCN(nn.Module):
    """Standard 2-layer GCN, Kipf & Welling baseline."""
    def __init__(self, nfeat, nhid, nclass, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(nfeat, nhid, cached=True)
        self.conv2 = GCNConv(nhid, nclass, cached=True)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)